# MedEvidence AI - Medical Research Knowledge Worker

## Project Overview

MedEvidence is a conversational Retrieval-Augmented Generation (RAG) assistant designed to help users explore and compare medical research on type 2 diabetes and GLP-1 medications.

The assistant retrieves relevant information from a medical knowledge base and generates answers grounded in research documents.

### Objectives
- Build a medical research knowledge base.
- Convert documents into embeddings and store them in chromaDB.
- Build a conversational RAG assistant using Langchain
- Display answers and retrieved sources in Gradio.
- Evaluate retrieval and answer quality.

**Scope:** Research and educational use only. Not for diagnosis or treatment decisions.

## Setting Up Our Knowledge Base

We will organize medical research documents by their source.

This will make it easier to load, process, and retrieve information later.

In [ ]:
from pathlib import Path

# Create our knowledge base
KNOWLEDGE_BASE = Path("knowledge-base")

folders = [
    "pubmed",
    "research-papers",
    "clinical-trials",
    "drug-labels"
]

for folder in folders:
    (KNOWLEDGE_BASE / folder).mkdir(parents=True, exist_ok=True)

print("Knowledge base created successfully!")

Knowledge base created successfully!


What this code does: Creates four folders where we'll store medical research data.

## Collecting Medical Research from PubMed

PubMed provides **biomedical research articles** and **abstracts**.

We will use the **NCBI API** to collect research on **type 2 diabetes** and **GLP-1 medications**.

Our goal is to create a **reliable medical knowledge base** for RAG.

In [4]:
# Importing the tools needed to request PubMed records, read XML data, and save them as JSON.
import requests
import json
import time
import xml.etree.ElementTree as ET
from pathlib import Path

### Searching PubMed

We will search for relevant medical research articles using NCBI's ESearch API.

The API returns unique PubMed IDs (PMIDs).

In [5]:
# It Connects to PubMed, searches for relevant medical research, and collects the unique IDs of up to 50 articles.
# SEARCH_QUERY: This is Search Query we send to PubMed.
#               Find Articles related to these terms (AND: Finding Articles Either one of the term mentioned.
#                                                      OR: means all connected groups must match)

# SEARCH_URL: This is the address of PubMed's ESearch API
# params: It is a Python dictionary that contains the instructions we want to send to PubMed.
#         db: Search the PubMed database
#         term: use the SEARCH_QUERY we created
#         retmax: Return a maximum of 50 Article IDs
#         retmode: Results the results in JSON format
#         sort: Sort results by relevance
#         tool: Identify our application as MedEvidenceAI

# response: Think of this as search button on PubMed, Python does this automatically.
#           request.get(): sends an HTTP GET Request
#           SEARCH_URL: tells where to send it
#           params: include our search settings
#           timeout=30: Python waits at most 30 sec for response before raising a timeout error

# pmids: It is PubMed Identifier.
#        It is a unique number assigned to record in PubMed.

SEARCH_QUERY = (
    '("type 2 diabetes" OR T2DM) AND '
    '(semaglutide OR tirzepatide OR "GLP-1") AND '
    '("randomized controlled trial" OR "systematic review" OR "meta-analysis") '
    'AND 2020:2026[dp]'
)

SEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": SEARCH_QUERY,
    "retmax": 50,
    "retmode": "json",
    "sort": "relevance",
    "tool": "MedEvidenceAI"
}

response = requests.get(SEARCH_URL, params=params, timeout=30)
response.raise_for_status()

search_results = response.json()
pmids = search_results["esearchresult"]["idlist"]

print("Total matching articles:", search_results["esearchresult"]["count"])           # Total no of matching articles in PubMed Site
print("Articles found:", len(pmids))                                                  # No of articles we requested (max 50)
print("First five PMIDs:", pmids[:5])                                                 # Display first 5 article IDs

Total matching articles: 1421
Articles found: 50
First five PMIDs: ['33667417', '38613667', '34170647', '39536238', '40055951']


### Downloading Research Records

We will use **NCBI's EFetch API** to retrieve article information.

Each record will contain the **title**, **abstract**, **publication information**, and **source identifiers** when available.

In [ ]:
FETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

params = {
    "db": "pubmed",
    "id": ",".join(pmids),
    "retmode": "xml",
    "tool": "MedEvidenceAI"
}

time.sleep(0.4)

response = requests.get(FETCH_URL, params=params, timeout=60)
response.raise_for_status()

root = ET.fromstring(response.content)

print("PubMed records downloaded successfully!")

# What this code does: Downloads the PubMed records in XML format.

PubMed records downloaded successfully!


### Preparing Our Research Dataset

We will extract important information from each article and save it in a structured format.

This will make document loading easier in our RAG pipeline.

In [9]:
# What this code does: Extracts the information we'll use in our knowledge base. 
#                      Records without abstracts are skipped.

def get_text(element):
    """Extract text from an XML element."""
    return "".join(element.itertext()).strip() if element is not None else ""


articles = []

for record in root.findall(".//PubmedArticle"):
    citation = record.find("MedlineCitation")
    article = citation.find("Article")

    pmid = get_text(citation.find("PMID"))
    title = get_text(article.find("ArticleTitle"))

    abstract_parts = article.findall(".//Abstract/AbstractText")
    abstract = " ".join(get_text(part) for part in abstract_parts)

    journal = get_text(article.find(".//Journal/Title"))

    year = get_text(article.find(".//PubDate/Year"))

    if not year:
        medline_date = get_text(article.find(".//PubDate/MedlineDate"))
        year = medline_date[:4] if medline_date else ""

    doi = ""
    for article_id in record.findall(".//PubmedData/ArticleIdList/ArticleId"):
        if article_id.get("IdType") == "doi":
            doi = get_text(article_id)
            break

    if pmid and title and abstract:
        articles.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "journal": journal,
            "year": year,
            "doi": doi,
            "source": "PubMed",
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        })

print("Articles with abstracts:", len(articles))



Articles with abstracts: 50


### Saving Our Knowledge Base

We will store the collected articles in a **JSON file**.

Later, LangChain will **load** this data and **convert** the research articles into searchable documents.

In [11]:
output_path = Path("knowledge-base/pubmed/pubmed_articles.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(articles, file, indent=4, ensure_ascii=False)

print("Saved to:", output_path)
print("Total articles saved:", len(articles))

Saved to: knowledge-base\pubmed\pubmed_articles.json
Total articles saved: 50


### Verifying Our Medical Research Dataset

We will load our saved JSON file and inspect the first article to confirm that our research data is stored correctly.

In [12]:
with open(output_path, "r", encoding="utf-8") as file:
    saved_articles = json.load(file)

print("Total saved articles:", len(saved_articles))

print("\nFirst Article:")
print("Title:", saved_articles[0]["title"])
print("Year:", saved_articles[0]["year"])
print("PMID:", saved_articles[0]["pmid"])
print("Abstract:", saved_articles[0]["abstract"][:300])

Total saved articles: 50

First Article:
Title: Semaglutide 2·4 mg once a week in adults with overweight or obesity, and type 2 diabetes (STEP 2): a randomised, double-blind, double-dummy, placebo-controlled, phase 3 trial.
Year: 2021
PMID: 33667417
Abstract: This trial assessed the efficacy and safety of the GLP-1 analogue once a week subcutaneous semaglutide 2·4 mg versus semaglutide 1·0 mg (the dose approved for diabetes treatment) and placebo for weight management in adults with overweight or obesity, and type 2 diabetes. This double-blind, double-du


## Loading Medical Research Documents

We have collected 50 research records from PubMed, stored them in a JSON file.

we will now convert these **records -> langchain Documents objects.**

Each document will contain:
- Page content: Article title and abstract
- metadata: PMID, journal, publication year, DOI, and source URL

This structure will help us **retrieve research information** and **provide source references** in our RAG assistant.

In [13]:
# importing required libraries
from langchain_core.documents import Document

## Creating Lngchain Documents

we will convert each PubMed research articles into a LangChain Document.

The article title and abstract will form the page content

Publication information will be stored as metadata, so our assistant can reference the original research.

In [14]:
# Creates an empty list called documents.
# Loops through our saved PubMed articles.
# Combines each articles title and abstract into searchable text.
# Stores the articles identifiers and publication details as metadata
# Creates the LangChain Document object
# Adds it to the list


documents = []

for article in saved_articles:

    content = (
        f"Title: {article['title']}\n\n"
        f"Abstract: {article['abstract']}"
    )

    metadata = {
        "pmid": article["pmid"],
        "title": article["title"],
        "journal": article["journal"],
        "year": article["year"],
        "doi": article["doi"],
        "source": article["source"],
        "url": article["url"]
    }

    document = Document(
        page_content=content,
        metadata=metadata
    )

    documents.append(document)

print("Total LangChain documents created:", len(documents))

Total LangChain documents created: 50


## Inspecting Our Langchain Documents

we will inspect the 1st document to verify that the research text and metadata have been stored correctly.

In [ ]:
# importing pprint for formatting the metadata output.
# Textwrap for Splitting and formatting between Title and Abstract

from pprint import pprint
from textwrap import fill

doc = documents[0]

print("DOCUMENT CONTENT:")
print("\n\n".join(fill(p, width=100) for p in doc.page_content.split("\n\n")))

print("\nDOCUMENT METADATA:")
pprint(doc.metadata, width=100, sort_dicts=False)

DOCUMENT CONTENT:
Title: Semaglutide 2·4 mg once a week in adults with overweight or obesity, and type 2 diabetes
(STEP 2): a randomised, double-blind, double-dummy, placebo-controlled, phase 3 trial.

Abstract: This trial assessed the efficacy and safety of the GLP-1 analogue once a week subcutaneous
semaglutide 2·4 mg versus semaglutide 1·0 mg (the dose approved for diabetes treatment) and placebo
for weight management in adults with overweight or obesity, and type 2 diabetes. This double-blind,
double-dummy, phase 3, superiority study enrolled adults with a body-mass index of at least 27 kg/m2
and glycated haemoglobin 7-10% (53-86 mmol/mol) who had been diagnosed with type 2 diabetes at least
180 days before screening. Patients were recruited from 149 outpatient clinics in 12 countries
across Europe, North America, South America, the Middle East, South Africa, and Asia. Patients were
randomly allocated (1:1:1) via an interactive web-response system and stratified by background
gluco

## Splitting Medical Research Documents into Chunks

Our PubMed knowledge base contains research articles with different abstract lengths.

Retrieving entire documents may introduce **irrelevant information** into the **context provided** to our language model.

To improve retrieval, we will divide **longer research documents** into **smaller text chunks**.

Each chunk will retain its original article metadata, allowing our assistant to reference the source of the retrieved information.

We will use LangChain's **RecursiveCharacterTextSplitter.**

In [26]:
# RecursiveCharacterTextSplitter: Tries to split text at paragraph breaks, newlines, and spaces 
#                                 before splitting at smaller character boundaries.

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,                    # Sets the maximum target length to 1000 characters.
    chunk_overlap=150,                  # Sets the target overlap between consecutive chunks to 150 characters.
    length_function=len
)

print("Text splitter configured successfully!")


Text splitter configured successfully!


In [34]:
# Create abstract-only documents without modifying the original documents.

# split_documents(): accepts the list of Langchain Document objects.
#                    split the page_content while copying the source metadata into each resulting chunk

abstract_documents = [
    Document(
        page_content=doc.page_content.partition("Abstract: ")[2],
        metadata=doc.metadata.copy()
    )
    for doc in documents
]

chunks = text_splitter.split_documents(abstract_documents)

print("Original documents:", len(documents))
print("Total chunks created:", len(chunks))

Original documents: 50
Total chunks created: 162


# Inspecting Our Medical Research Chunks

Before using the chunks in our RAG pipeline, we need to examine their content and source information.

This inspection helps us understand how the original research text was divided and whether the article metadata remains available.

Preserving source information is especially important for evidence-grounded research applications.

In [36]:
from textwrap import fill
from pprint import pprint

chunk = chunks[0]

print("CHUNK CONTENT:")
print("Title:", fill(chunk.metadata["title"], width=100))

print("\nAbstract:")
print(fill(chunk.page_content, width=100))

print("\nCHUNK METADATA:")
pprint(chunk.metadata, width=100, sort_dicts=False)

print("\nCHUNK LENGTH:", len(chunk.page_content))

CHUNK CONTENT:
Title: Semaglutide 2·4 mg once a week in adults with overweight or obesity, and type 2 diabetes (STEP 2): a
randomised, double-blind, double-dummy, placebo-controlled, phase 3 trial.

Abstract:
This trial assessed the efficacy and safety of the GLP-1 analogue once a week subcutaneous
semaglutide 2·4 mg versus semaglutide 1·0 mg (the dose approved for diabetes treatment) and placebo
for weight management in adults with overweight or obesity, and type 2 diabetes. This double-blind,
double-dummy, phase 3, superiority study enrolled adults with a body-mass index of at least 27 kg/m2
and glycated haemoglobin 7-10% (53-86 mmol/mol) who had been diagnosed with type 2 diabetes at least
180 days before screening. Patients were recruited from 149 outpatient clinics in 12 countries
across Europe, North America, South America, the Middle East, South Africa, and Asia. Patients were
randomly allocated (1:1:1) via an interactive web-response system and stratified by background
glucose-

In [37]:
# Check that all 50 original research articles are represented.
original_pmids = {doc.metadata["pmid"] for doc in documents}
chunk_pmids = {chunk.metadata["pmid"] for chunk in chunks}

# Verify that every chunk contains text, source information,
# and stays within our configured maximum size.
assert original_pmids == chunk_pmids
assert all(chunk.page_content.strip() for chunk in chunks)
assert all(chunk.metadata.get("title") for chunk in chunks)
assert all(chunk.metadata.get("url") for chunk in chunks)
assert all(len(chunk.page_content) <= 1000 for chunk in chunks)

print("Step 4 verification passed!")
print("Unique research articles:", len(chunk_pmids))
print("Total chunks created:", len(chunks))
print("Largest chunk:", max(len(chunk.page_content) for chunk in chunks))

Step 4 verification passed!
Unique research articles: 50
Total chunks created: 162
Largest chunk: 1000


## Creating Embeddings and Storing Documents in ChromaDB

We have successfully divided 50 PubMed research articles into 162 chunks.

Our next goal is to make these chunks searchable.

We will use:
- OpenAI Embeddings to convert research text into numerical vectors.
- ChromaDB to store these vectors along with their text and source information.
- LangChain to connect our documents, embeddings, and vector database.

This will allow MedEvidence AI to find relevant research even when
a user's question does not contain the exact words used in an article.

In [38]:
# Loading API Key
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "OpenAI API key not found. Check your .env file."
print("OpenAI API key loaded successfully!")

OpenAI API key loaded successfully!


## Initializing Our Embedding Model

An embedding is a **numerical representation of text.**

Text with **similar meaning** generally **produce vectors** that are **closer to each other.**

Using **text-embedding-3-small** model to generate embeddings for our medical research chunks.

Same model will be used for searching the database.

In [39]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")   # OpenAIEmbeddings(): Connects LangChain to OpenAI's embedding model.

print("Embedding model initialized successfully!")

Embedding model initialized successfully!


In [40]:
# Testing Embedding Model
test_text = "Semaglutide treatment for type 2 diabetes"

test_embedding = embeddings.embed_query(test_text)

print("Embedding dimensions:", len(test_embedding))
print("First five values:", test_embedding[:5])

Embedding dimensions: 1536
First five values: [-0.00939178466796875, 0.0085296630859375, 0.01873779296875, -0.00191497802734375, -0.052886962890625]


### Creating Our Vector Database

We will create a ChromaDb collection to store our medical research.

Each stored record will contain:
- The Research chunk
- The embedding
- The original article metadata

and we will save the database locally for reusing.

In [41]:
from langchain_chroma import Chroma

DB_PATH = "chroma_db"
COLLECTION_NAME = "medevidence_pubmed"

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

print("ChromaDB initialized successfully!")
print("Database location:", DB_PATH)

ChromaDB initialized successfully!
Database location: chroma_db


### Storing Our Medical Research Chunks

We will now store our 162 research chunks in ChromaDB.

Langchain will generate an embedding for each chunk and store it together with original text and metadata.

We will also assign a unique ID to every chunk so that rerunning the notebook does not create duplicate records.

In [42]:
from collections import defaultdict

# defaultdict(int): Starts a counter at zero for each PubMed article.
# PMID: Identifies the original research article.
# chunk_number: Identifies a particular chunk within that article.
# chunk_ids: Stores the unique identifiers used by ChromaDB.

article_counts = defaultdict(int)
chunk_ids = []

for chunk in chunks:
    pmid = chunk.metadata["pmid"]
    article_counts[pmid] += 1

    chunk_ids.append(
        f"pubmed_{pmid}_chunk_{article_counts[pmid]}"
    )

print("Total chunks:", len(chunks))
print("Unique IDs:", len(set(chunk_ids)))

Total chunks: 162
Unique IDs: 162


In [43]:
# add_documents(): Generates embeddings and stores the documents.
# documents: Our 162 medical research chunks.
# ids: Stable identifiers for the stored chunks.

vector_store.add_documents(
    documents=chunks,
    ids=chunk_ids
)

print("Research chunks stored successfully!")

Research chunks stored successfully!


### Verifying Our Vector Database

We will verify: 
- whether all 162 research chunks were stored
- stored embeddings ahve the expected dimensions.

These checks help confirm that our vector database is ready for semantic search.

In [44]:
# count(): returns the no of stored vectors.
# get(): retrieves information from the ChromaDB collection
# include=["embeddings"]: requests the humericl embedding vectors.

stored_count = vector_store._collection.count()

sample = vector_store._collection.get(limit=1, include=["embeddings"])

embedding_dimensions = len(sample["embeddings"][0])

assert stored_count == len(chunks), "Stored chunk count does not match."
assert embedding_dimensions == 1536, "Embedding dimensions do not match."

print("Step 5 verification passed!")
print("Stored research chunks:", stored_count)
print("Embedding dimensions:", embedding_dimensions)

Step 5 verification passed!
Stored research chunks: 162
Embedding dimensions: 1536


### Testing Semantic Search

Our research chunks are now stored in ChromaDB.

We will ask a sample medical research question and retrieve the three most relevant chunks.

This will help us confirm that our vector database can search research documents based on their meaning.

In [45]:
# similarity_search(): Searches for chunks similar to our question.
# query: The research question we want to search.
# k=3: Returns the three most similar chunks.

from textwrap import fill

question = "What are the effects of semaglutide on weight loss in type 2 diabetes?"

results = vector_store.similarity_search(query=question, k=3)

for i, result in enumerate(results, start=1):
    print(f"\nRESULT {i}")
    print("Title:", result.metadata["title"])
    print("PMID:", result.metadata["pmid"])
    print("Source:", result.metadata["url"])
    print("\nContent:")
    print(fill(result.page_content, width=100))


RESULT 1
Title: Long-Term Efficacy and Safety of Once-Weekly Semaglutide for Weight Loss in Patients Without Diabetes: A Systematic Review and Meta-Analysis of Randomized Controlled Trials.
PMID: 38679221
Source: https://pubmed.ncbi.nlm.nih.gov/38679221/

Content:
Semaglutide, a glucagon-like peptide-1 receptor agonist, has demonstrated clinically important
weight loss effects in patients with type 2 diabetes. However, its effects on sustained weight loss
in patients without diabetes remains unclear. Our objective was to examine the long-term efficacy
and safety of semaglutide use for weight loss in patients with overweight/obesity and without
diabetes. MEDLINE, EMBASE, and the Cochrane Libraries were systematically searched to identify
randomized controlled trials that randomized participants with overweight/obesity and without
diabetes to once-weekly 2.4 mg subcutaneous semaglutide versus placebo, with a follow-up of at least
68 weeks. The primary outcome was a change in relative bo

## Building Our Medical Research Retriever

We have stored 162 medical research chunks in ChromaDB.

Now, we will create a retriever that searches our knowledge base and finds relevant research based on a user's question.

A retriever acts like a search engine for our medical documents.

Instead of manually searching ChromaDB, we can use the retriever to automatically find the most relevant research chunks.

These retrieved chunks will later help our AI assistant generate answers supported by medical research.

In [46]:
# as_retriever(): Converts our ChromaDB vector store into a LangChain retriever.

retriever = vector_store.as_retriever(
    search_type="similarity",                    # Finds documents whose embeddings are most similar to the question.
    search_kwargs={"k": 5}                       # Retrieves the five most similar research chunks.
)

print("Medical research retriever created successfully!")

Medical research retriever created successfully!


### Retrieving Medical Research

We will test our retriever using a question about semaglutide and type 2 diabetes.

The retriever will convert the question into an embedding and compare it with the research embeddings stored in ChromaDB.

It will return the five most similar research chunks.

At this stage, we are retrieving information, not generating an answer using a language model.

In [47]:
# invoke(): Sends our research question to the retriever.
#           The retriever returns a list of relevant LangChain Documents.
# retrieved_docs: Stores the research chunks returned by ChromaDB.

question = "What are the effects of semaglutide on weight loss in type 2 diabetes?"

retrieved_docs = retriever.invoke(question)

print("Research chunks retrieved:", len(retrieved_docs))

Research chunks retrieved: 5


In [48]:
# Creating Reusbale Research Search Function
# Accepts a medical research question and returns relevant chunks.
# 
def retrieve_research(query):  
    return retriever.invoke(query)             # retriever.invoke(): Searches ChromaDB for matching research.


print("Research retrieval function created successfully!")

Research retrieval function created successfully!


In [50]:
# Testing Reusable Function
test_question = "How does tirzepatide affect blood sugar control in type 2 diabetes?"

test_results = retrieve_research(test_question)

print("Research chunks retrieved:", len(test_results))

for i, doc in enumerate(test_results, start=1):
    print(f"\nRESULT {i}")
    print("Title:", doc.metadata["title"])
    print("PMID:", doc.metadata["pmid"])

Research chunks retrieved: 5

RESULT 1
Title: Effect of Subcutaneous Tirzepatide vs Placebo Added to Titrated Insulin Glargine on Glycemic Control in Patients With Type 2 Diabetes: The SURPASS-5 Randomized Clinical Trial.
PMID: 35133415

RESULT 2
Title: Management of type 2 diabetes with the dual GIP/GLP-1 receptor agonist tirzepatide: a systematic review and meta-analysis.
PMID: 35579691

RESULT 3
Title: Efficacy and safety of a novel dual GIP and GLP-1 receptor agonist tirzepatide in patients with type 2 diabetes (SURPASS-1): a double-blind, randomised, phase 3 trial.
PMID: 34186022

RESULT 4
Title: Tirzepatide, a dual GIP/GLP-1 receptor co-agonist for the treatment of type 2 diabetes with unmatched effectiveness regrading glycaemic control and body weight reduction.
PMID: 36050763

RESULT 5
Title: Tirzepatide for Obesity Treatment and Diabetes Prevention.
PMID: 39536238


### Verifying Our Medical Research Retriever

Before building our question-answering system, we need to confirm that our retriever works correctly.

We will verify that:
- Research documents are successfully retrieved.
- Every retrieved document contains research text.
- Every document includes its original article information.
- The number of returned chunks matches our retrieval settings.

These checks confirm that our retrieval pipeline is ready for the next stage of MedEvidence AI.

In [51]:
# assert: Checks that each required condition is satisfied.
# all(): Confirms that every retrieved document passes a check.

# We verify the content and source metadata of our retrieved chunks.

assert len(retrieved_docs) == 5
assert len(test_results) == 5

assert all(doc.page_content.strip() for doc in retrieved_docs)
assert all(doc.metadata.get("pmid") for doc in retrieved_docs)
assert all(doc.metadata.get("title") for doc in retrieved_docs)
assert all(doc.metadata.get("url") for doc in retrieved_docs)

assert all(doc.page_content.strip() for doc in test_results)
assert all(doc.metadata.get("pmid") for doc in test_results)

print("Step 6 verification passed!")
print("Semaglutide query results:", len(retrieved_docs))
print("Tirzepatide query results:", len(test_results))

Step 6 verification passed!
Semaglutide query results: 5
Tirzepatide query results: 5


## Building Our RAG Question-Answering System

We have created a **retriever** that finds **relevant medical research** from our **ChromaDB knowledge base**.

However, retrieving documents is not the same as answering a question.

We will now **connect** our **retriever to a Large Language Model (LLM)** using **Retrieval-Augmented Generation (RAG).**

Our RAG system will:
- Search the knowledge base for relevant research.
- Provide the retrieved evidence to the language model.
- Generate an answer based on that evidence.
- Include references to the original PubMed articles.

MedEvidence AI is designed for research and educational purposes, not personalized medical advice.

In [52]:
# Initializing Our Model
# Retriever finds relevant research, but it cannot explain findings in conversational format

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0              # Reduces randomness for more consistent responses.
)

print("Model initialized successfully!")

Model initialized successfully!


## Creating Research Prompt

For MedEvidence AI, we want answers supported by the research
stored in our knowledge base.

We will create a prompt that instructs the model to:
- Answer using only the retrieved research.
- Avoid inventing medical findings.
- Acknowledge when the evidence is insufficient.
- Reference the PubMed sources used in its answer.

This helps make our assistant more transparent and reliable.

In [53]:
from langchain_core.prompts import ChatPromptTemplate

# ChatPromptTemplate.from_messages(): Creates a reusable prompt containing system instructions and the user's research question.

# {context}: Will contain the retrieved PubMed research.
# {question}: Will contain the user's question.

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are MedEvidence AI, a medical research assistant.

Answer the user's question using ONLY the provided research context.

Instructions:
1. Use only information supported by the retrieved research.
2. Do not invent findings, statistics, or conclusions.
3. If the context is insufficient, clearly say so.
4. Explain the findings in clear, understandable language.
5. Cite supporting studies using their PMID in the format [PMID: 12345678].
6. Do not provide personalized medical advice, diagnoses, or treatment recommendations.
7. Distinguish research findings from uncertainty or limitations.

Research Context:
{context}
"""
    ),
    (
        "human",
        "{question}"
    )
])

print("Medical research prompt created successfully!")

Medical research prompt created successfully!


### Preparing Research Context

In [54]:
# format_research_context(): Converts retrieved LangChain Documents into a single text block.
# enumerate(): Assigns a number to each retrieved research chunk.
# metadata: Provides the article title, PMID, and source URL.
# "\n\n".join(): Separates the research records with blank lines.

def format_research_context(docs):
    formatted_docs = []

    for i, doc in enumerate(docs, start=1):

        formatted_doc = (
            f"Research Source {i}\n"
            f"Title: {doc.metadata['title']}\n"
            f"PMID: {doc.metadata['pmid']}\n"
            f"URL: {doc.metadata['url']}\n"
            f"Research Content: {doc.page_content}"
        )

        formatted_docs.append(formatted_doc)

    return "\n\n".join(formatted_docs)


print("Research context formatting function created successfully!")

Research context formatting function created successfully!


In [55]:
# Reuse the retrieved documents from Step 6.
# Display the beginning of the formatted context.

context = format_research_context(retrieved_docs)

print(context[:800])

Research Source 1
Title: Long-Term Efficacy and Safety of Once-Weekly Semaglutide for Weight Loss in Patients Without Diabetes: A Systematic Review and Meta-Analysis of Randomized Controlled Trials.
PMID: 38679221
URL: https://pubmed.ncbi.nlm.nih.gov/38679221/
Research Content: Semaglutide, a glucagon-like peptide-1 receptor agonist, has demonstrated clinically important weight loss effects in patients with type 2 diabetes. However, its effects on sustained weight loss in patients without diabetes remains unclear. Our objective was to examine the long-term efficacy and safety of semaglutide use for weight loss in patients with overweight/obesity and without diabetes. MEDLINE, EMBASE, and the Cochrane Libraries were systematically searched to identify randomized controlled trials that rando


### Creating Our Research Answer Function

We will now combine the retriever, research context, prompt, and language model into one reusable function.

When a user asks a question, our system will:
1. Retrieve relevant research from ChromaDB.
2. Prepare the retrieved evidence for the language model.
3. Generate an answer based on that evidence.
4. Return the answer together with its sources.

This is the main question-answering workflow of MedEvidence AI.

In [57]:
# answer_question(): Accepts a research question and generates an evidence-grounded answer.
# retrieve_research(): Finds relevant chunks from ChromaDB.
# format_research_context(): Prepares the research for the LLM.
# prompt.invoke(): Inserts the question and research into our prompt.
# llm.invoke(): Generates the answer using the supplied evidence.
#               Returns the generated answer and the retrieved source documents.

def answer_question(question):

    docs = retrieve_research(question)

    if not docs:
        return "No relevant research was found in the knowledge base.", []

    context = format_research_context(docs)

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = llm.invoke(messages)

    return response.content, docs


print("MedEvidence AI question-answering function created successfully!")

MedEvidence AI question-answering function created successfully!


### Testing Our Medical Research Assistant

Our RAG pipeline is now connected.

We will ask a question about semaglutide and weight loss in patients with type 2 diabetes.

MedEvidence AI will retrieve relevant PubMed research, generate an answer, and reference supporting studies.

We will inspect the response to see whether it is grounded in the retrieved evidence.

In [58]:
# Ask a medical research question.
# answer_question() returns both the answer and retrieved documents.

question = "What does research show about semaglutide and weight loss in type 2 diabetes?"

answer, sources = answer_question(question)

print("QUESTION:")
print(question)

print("\nMEDEVIDENCE AI ANSWER:")
print(answer)

QUESTION:
What does research show about semaglutide and weight loss in type 2 diabetes?

MEDEVIDENCE AI ANSWER:
Research indicates that semaglutide, a glucagon-like peptide-1 receptor agonist, is effective for weight loss in patients with type 2 diabetes. In a phase 3 trial, semaglutide 2.4 mg once a week achieved a significant and clinically meaningful decrease in body weight compared to placebo [PMID: 33667417]. 

However, the specific details regarding the extent of weight loss or the long-term effects in this population were not provided in the context. Therefore, while semaglutide is shown to be beneficial for weight management in individuals with type 2 diabetes, further details on its long-term efficacy and safety in this group are not available from the provided research.


### Displaying Supporting Research Sources

A research assistant should make it easy to verify where its information came from.

We will display the PubMed articles retrieved for our answer, including their titles and source links.

This allows users to inspect the original research rather than relying only on the AI-generated explanation.

In [59]:
# Display each unique PubMed article only once.
# A single article may have multiple chunks in the retrieved results.

seen_pmids = set()

print("RETRIEVED RESEARCH SOURCES:")

for doc in sources:
    pmid = doc.metadata["pmid"]

    if pmid not in seen_pmids:
        print("\nTitle:", doc.metadata["title"])
        print("PMID:", pmid)
        print("Source:", doc.metadata["url"])

        seen_pmids.add(pmid)

RETRIEVED RESEARCH SOURCES:

Title: Long-Term Efficacy and Safety of Once-Weekly Semaglutide for Weight Loss in Patients Without Diabetes: A Systematic Review and Meta-Analysis of Randomized Controlled Trials.
PMID: 38679221
Source: https://pubmed.ncbi.nlm.nih.gov/38679221/

Title: Efficacy and Safety of Semaglutide for Weight Loss in Obesity Without Diabetes: A Systematic Review and Meta-Analysis.
PMID: 36578889
Source: https://pubmed.ncbi.nlm.nih.gov/36578889/

Title: Semaglutide 2·4 mg once a week in adults with overweight or obesity, and type 2 diabetes (STEP 2): a randomised, double-blind, double-dummy, placebo-controlled, phase 3 trial.
PMID: 33667417
Source: https://pubmed.ncbi.nlm.nih.gov/33667417/

Title: A systematic review of the effect of semaglutide on lean mass: insights from clinical trials.
PMID: 38629387
Source: https://pubmed.ncbi.nlm.nih.gov/38629387/


### Verifying Our Question-Answering System

Before moving forward, we need to confirm that our RAG pipeline successfully connects retrieval and generation.

We will verify that:
- The assistant generates a non-empty answer.
- Research documents are retrieved.
- Retrieved documents contain valid PubMed identifiers.
- Retrieved documents include source URLs.

These checks confirm that our system works technically.

In [60]:
# Validate the generated answer and retrieved research sources.

assert isinstance(answer, str) and answer.strip()
assert len(sources) > 0

assert all(doc.page_content.strip() for doc in sources)
assert all(doc.metadata.get("pmid") for doc in sources)
assert all(doc.metadata.get("url") for doc in sources)

print("Step 7 verification passed!")
print("Research chunks retrieved:", len(sources))
print("Unique PubMed articles:", len({doc.metadata["pmid"] for doc in sources}))
print("Answer length:", len(answer), "characters")

Step 7 verification passed!
Research chunks retrieved: 5
Unique PubMed articles: 4
Answer length: 678 characters


## Re-Ranking

## Improving Research Retrieval with LLM based training

Our Retriever finds research chunks using similarity.

However, the chunk that is semantically similar to a question may not always contain the most useful evidence for answering it.

To improve retrieval, we will introduce the LLM-based reranking.

Now, Our system will:
 - Retrieve 8 potentially relevant research chunks from ChromaDB.
 - Ask anLLM to evaluate their relavnce to the question.
 - Reorder the chunks from most relevant to least relevant.
 - Select the top 5 chuks for answer generation

 This will help MedEvidence AI prioritize research evidence that is more directly related to the user question.

### Retrieving Candidate Research Chunks

Our existing retriever returns five research chunks.

For reranking, we will retrieve eight chunks to give the language model a larger set of potential evidence.

The LLM will then select and reorder the five most relevant chunks from these candidates.

In [61]:
# Create a separate retriever that returns 8 candidate chunks.
# Our original Step 6 retriever remains unchanged.

candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

print("Candidate retriever created successfully!")

Candidate retriever created successfully!


In [62]:
### Creating Our Reranking Prompt
from langchain_core.prompts import ChatPromptTemplate

# The LLM receives the research question and numbered chunks.
# It must return only the chunk numbers, ordered by relevance.

rerank_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a medical research retrieval assistant.

Rank the provided research chunks by relevance to the question.

Prioritize:
- Research directly addressing the question.
- Findings and outcomes relevant to the question.
- Specific evidence over general background information.

Do not invent findings or use outside knowledge.

Return ONLY a JSON object in this format:
{{"ranking": [1, 3, 2]}}

Include every provided chunk number exactly once.
"""
    ),
    (
        "human",
        """
Question:
{question}

Research Chunks:
{context}
"""
    )
])

print("Reranking prompt created successfully!")

Reranking prompt created successfully!


### Creating Our Research Reranking Function

We will create a reusable function that sends the candidate research chunks to the language model.

The model will return the order in which the chunks should be ranked.

Our function will check the returned ranking before using it.

In [63]:
import json

def rerank_research(question, docs, top_k=5):

    if not docs:
        return []

    # Number each chunk so the LLM can identify it.
    context = "\n\n".join(
        f"Chunk {i}\n"
        f"Title: {doc.metadata['title']}\n"
        f"Content: {doc.page_content}"
        for i, doc in enumerate(docs, start=1)
    )

    # Send the question and candidate chunks to the LLM.
    messages = rerank_prompt.invoke({
        "question": question,
        "context": context
    })

    try:
        response = llm.invoke(messages)

        # Convert the JSON response into a Python dictionary.
        ranking = json.loads(response.content)["ranking"]

        # Verify that every chunk appears exactly once.
        expected = set(range(1, len(docs) + 1))

        if (
            not isinstance(ranking, list)
            or len(ranking) != len(docs)
            or any(type(i) is not int for i in ranking)
            or set(ranking) != expected
        ):
            raise ValueError("Invalid reranking response")

        # Convert the 1-based ranking into Python list indices.
        reranked_docs = [docs[i - 1] for i in ranking]

        return reranked_docs[:top_k]

    except (ValueError, KeyError, TypeError, AttributeError):
        # Keep the original similarity ranking if parsing fails.
        print("Invalid reranking output. Using original retrieval order.")
        return docs[:top_k]


print("Research reranking function created successfully!")

Research reranking function created successfully!


### Testing Research Reranking

We will test reranking using a question about semaglutide and weight loss in patients with type 2 diabetes.

First, ChromaDB will retrieve eight candidate chunks.

Then, the LLM will reorder those chunks based on their relevance to the research question.

We will compare the original order with the reranked order.

In [64]:
question = "What does research show about semaglutide and weight loss in type 2 diabetes?"

# Retrieve 8 candidate chunks.
candidate_docs = candidate_retriever.invoke(question)

# Rerank the candidates and keep the top 5.
reranked_docs = rerank_research(
    question,
    candidate_docs,
    top_k=5
)

print("Candidate chunks:", len(candidate_docs))
print("Reranked chunks:", len(reranked_docs))

Candidate chunks: 8
Reranked chunks: 5


### Comparing Retrieval Results

We will compare the **original ChromaDB ranking** with the **ranking produced by our language model.**

This will help us understand **whether the LLM changes the order of retrieved research.**

A change in ranking does not automatically mean retrieval has improved. We will measure that during evaluation.

In [65]:
print("ORIGINAL CHROMADB RANKING:")

for i, doc in enumerate(candidate_docs, start=1):
    print(f"{i}. PMID: {doc.metadata['pmid']}")
    print("   Title:", doc.metadata["title"][:100])


print("\nLLM RERANKED RESULTS:")

for i, doc in enumerate(reranked_docs, start=1):
    print(f"{i}. PMID: {doc.metadata['pmid']}")
    print("   Title:", doc.metadata["title"][:100])

ORIGINAL CHROMADB RANKING:
1. PMID: 38679221
   Title: Long-Term Efficacy and Safety of Once-Weekly Semaglutide for Weight Loss in Patients Without Diabete
2. PMID: 36578889
   Title: Efficacy and Safety of Semaglutide for Weight Loss in Obesity Without Diabetes: A Systematic Review 
3. PMID: 33667417
   Title: Semaglutide 2·4 mg once a week in adults with overweight or obesity, and type 2 diabetes (STEP 2): a
4. PMID: 38629387
   Title: A systematic review of the effect of semaglutide on lean mass: insights from clinical trials.
5. PMID: 36578889
   Title: Efficacy and Safety of Semaglutide for Weight Loss in Obesity Without Diabetes: A Systematic Review 
6. PMID: 38330988
   Title: Efficacy and safety of once weekly semaglutide 2·4 mg for weight management in a predominantly east 
7. PMID: 40550013
   Title: Semaglutide in Adults with Type 1 Diabetes and Obesity.
8. PMID: 38330988
   Title: Efficacy and safety of once weekly semaglutide 2·4 mg for weight management in a predominantly

You may notice that some articles appear more than once because different chunks can belong to the same PubMed article.

### Integrating Reranking with Our RAG Pipeline

Our existing question-answering system uses research retrieved directly from ChromaDB.

We will now create an improved version that includes LLM-based reranking before generating an answer.

This will allow MedEvidence AI to use the highest-ranked research chunks as evidence for its responses.

We will keep the original answer function unchanged so we can compare both approaches later.

In [66]:
# This function extends our Step 7 workflow with reranking.
# It reuses our existing prompt, context formatter, and LLM.

def answer_question_reranked(question):

    candidates = candidate_retriever.invoke(question)

    if not candidates:
        return "No relevant research was found in the knowledge base.", []

    docs = rerank_research(question, candidates, top_k=5)

    context = format_research_context(docs)

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = llm.invoke(messages)

    return response.content, docs


print("Reranked RAG pipeline created successfully!")

Reranked RAG pipeline created successfully!


### Testing Our Reranked RAG Assistant

We will now ask MedEvidence AI a research question using our improved retrieval pipeline.

The system will retrieve candidate research chunks, rerank them, and generate an answer using the selected evidence.

We will inspect the answer and its source references.

In [67]:
question = "How does tirzepatide affect blood sugar control in type 2 diabetes?"

answer, sources = answer_question_reranked(question)

print("QUESTION:")
print(question)

print("\nMEDEVIDENCE AI ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

seen_pmids = set()

for doc in sources:
    pmid = doc.metadata["pmid"]

    if pmid not in seen_pmids:
        print("Title:", doc.metadata["title"])
        print("PMID:", pmid)
        print("URL:", doc.metadata["url"])
        print()

        seen_pmids.add(pmid)

QUESTION:
How does tirzepatide affect blood sugar control in type 2 diabetes?

MEDEVIDENCE AI ANSWER:
Tirzepatide has been shown to significantly improve blood sugar control in patients with type 2 diabetes. In clinical trials, the addition of tirzepatide to insulin glargine treatment resulted in statistically significant reductions in HbA1c levels, which is a measure of average blood sugar over time. For example, patients receiving tirzepatide experienced mean HbA1c changes from baseline of -2.11% with the 5 mg dose, -1.53% with the 10 mg dose, and -1.47% with the 15 mg dose, compared to a -0.86% change with placebo. Additionally, a higher percentage of patients treated with tirzepatide achieved HbA1c levels of less than 7% compared to those on placebo (85%-90% vs 34%) [PMID: 35133415].

Overall, tirzepatide has demonstrated robust improvements in glycemic control, making it an effective option for managing blood sugar levels in individuals with type 2 diabetes [PMID: 36050763].

RETR

## Building the MedEvidence AI Interface

We have successfully built a RAG system that retrieves medical research, reranks relevant documents, and generates answers.

However, users currently need to run Python code to ask questions.

We will use Gradio to create a simple web interface where users can:

- Enter a medical research question.
- Generate an answer based on PubMed research.
- View the supporting research articles and their source links.

This will make MedEvidence AI easier to use for both technical and non-technical users.

### Connecting Our RAG System to Gradio

Our RAG system already has a function called **answer_question_reranked().**

This function retrieves research, reranks the results, and generates an answer using the selected evidence.

We will create a new function that **formats the answer** and **PubMed sources for display in our web interface.**

We will also **remove duplicate articles** from the source list.

In [68]:
def gradio_answer(question):

    # Check whether the user entered a valid question.
    if not question.strip():
        return "Please enter a medical research question.", ""

    # Generate an answer using our Step 8 RAG pipeline.
    answer, sources = answer_question_reranked(question)

    # Format the retrieved research sources.
    source_list = []
    seen_pmids = set()

    for doc in sources:

        pmid = doc.metadata["pmid"]

        if pmid not in seen_pmids:

            title = doc.metadata["title"]
            url = doc.metadata["url"]

            source_list.append(
                f"- **{title}**  \n"
                f"  PMID: {pmid} | [View on PubMed]({url})"
            )

            seen_pmids.add(pmid)

    formatted_sources = "\n\n".join(source_list)

    return answer, formatted_sources


print("Gradio interface function created successfully!")

Gradio interface function created successfully!


### Creating Gradio Interface

In [71]:
import gradio as gr
from html import escape
from urllib.parse import urlparse

# Format the answer and supporting research for our interface.
def research_interface(question, retrieval_method):

    if not question or not question.strip():
        return "Please enter a medical research question.", "", "0 studies"

    # Select the retrieval pipeline chosen by the user.
    if retrieval_method == "Semantic Search":
        answer, sources = answer_question(question)
    else:
        answer, sources = answer_question_reranked(question)

    # Remove duplicate PubMed articles.
    seen_pmids = set()
    source_items = []

    for doc in sources:
        pmid = str(doc.metadata.get("pmid", ""))

        if pmid in seen_pmids:
            continue

        seen_pmids.add(pmid)

        title = escape(str(doc.metadata.get("title", "Untitled research article")))

        # Construct the link from the PubMed identifier.
        # Only numerical PMIDs are allowed in the generated URL.
        if pmid.isdigit():
            url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            link = (
                f'<a href="{url}" target="_blank" rel="noopener noreferrer">'
                'View on PubMed ↗</a>'
            )
        else:
            link = ""

        source_items.append(
            f"""
            <div class="research-item">
                <div class="research-title">{title}</div>
                <div class="research-bottom">
                    <span>PMID: {escape(pmid)}</span>
                    {link}
                </div>
            </div>
            """
        )

    sources_html = "".join(source_items)

    return answer, sources_html, f"{len(seen_pmids)} studies"


print("Gradio interface function created successfully!")

Gradio interface function created successfully!


In [103]:
from pathlib import Path

# Read our CSS file from the static folder.
css_path = Path("static/style.css")

assert css_path.exists(), "CSS file not found. Check the file path."

carbon_css = css_path.read_text(encoding="utf-8")

print("CSS File loaded successfully!")

CSS File loaded successfully!


In [104]:
# Close the previous Gradio app before rebuilding it.
if "demo" in globals():
    try:
        demo.close()
    except Exception:
        pass


with gr.Blocks(
    title="MedEvidence AI",
    theme=gr.themes.Base(
        primary_hue="neutral",
        neutral_hue="neutral"
    ),
    css=carbon_css
) as demo:

    # ---------- HEADER ----------
    gr.HTML(
        """
        <div id="app-header">
            <h2>MedEvidence AI</h2>
            <p>Medical Research Assistant</p>
        </div>
        """
    )

    # ---------- RESEARCH WORKSPACE ----------
    with gr.Column(elem_id="research-workspace"):

        gr.HTML(
            """
            <div id="welcome-section">
                <h1>What can I help you research?</h1>
                <p>
                    Explore clinical studies and discover
                    evidence-backed insights.
                </p>
            </div>
            """
        )

        # ---------- SINGLE SEARCH COMPOSER ----------

        # Column avoids Gradio Group's outer border and background.
        with gr.Column(elem_id="research-input"):

            question_input = gr.Textbox(
                placeholder="Ask MedEvidence AI...",
                lines=1,
                max_lines=6,
                show_label=False,
                container=False,
                elem_id="question-input"
            )

            # Retrieval selector and send button share one bottom row.
            with gr.Row(
                elem_id="input-toolbar",
                equal_height=True
            ):

                retrieval_method = gr.Dropdown(
                    choices=[
                        "Semantic Search",
                        "LLM Reranking"
                    ],
                    value="LLM Reranking",
                    show_label=False,
                    container=False,
                    elem_id="retrieval-dropdown",
                    scale=0,
                    min_width=190
                )

                send_button = gr.Button(
                    "↑",
                    elem_id="send-button",
                    scale=0,
                    min_width=36
                )

        # ---------- EXAMPLE QUESTIONS ----------
        with gr.Row(
            elem_id="example-questions",
            equal_height=True
        ):

            example_1 = gr.Button(
                "Semaglutide research",
                elem_classes=["example-button"],
                scale=0,
                min_width=0
            )

            example_2 = gr.Button(
                "Tirzepatide studies",
                elem_classes=["example-button"],
                scale=0,
                min_width=0
            )

            example_3 = gr.Button(
                "Compare GLP-1 therapies",
                elem_classes=["example-button"],
                scale=0,
                min_width=0
            )

    # ---------- RESEARCH RESULTS ----------
    with gr.Column(
        visible=False,
        elem_id="results-section"
    ) as results_section:

        gr.HTML(
            """
            <div id="answer-heading">
                <h3>Evidence summary</h3>
            </div>
            """
        )

        answer_output = gr.Markdown(
            elem_id="answer-output",
            container=False
        )

        gr.HTML(
            """
            <hr style="
                border: 0;
                border-top: 1px solid #303030;
                margin: 28px 0;
            ">
            """
        )

        # ---------- SUPPORTING RESEARCH ----------
        with gr.Row(
            elem_id="sources-header",
            equal_height=True
        ):

            gr.HTML(
                """
                <div id="sources-heading">
                    <h3>Supporting research</h3>
                </div>
                """
            )

            source_count = gr.Markdown(
                value="0 studies",
                elem_id="source-count",
                container=False
            )

        sources_output = gr.HTML()

    # ---------- FOOTER ----------
    gr.HTML(
        """
        <div id="app-footer">
            For research and educational purposes only.
            Not intended for medical diagnosis or treatment advice.
        </div>
        """
    )

    # ---------- RESEARCH FUNCTION ----------
    def submit_research(question, method):

        if not question or not question.strip():
            return (
                "Please enter a medical research question.",
                "",
                "0 studies",
                gr.update(visible=True)
            )

        # Use the existing MedEvidence AI RAG pipeline.
        answer, sources, count = research_interface(
            question,
            method
        )

        return (
            answer,
            sources,
            count,
            gr.update(visible=True)
        )

    # ---------- SUBMIT BUTTON ----------
    send_button.click(
        fn=submit_research,
        inputs=[question_input, retrieval_method],
        outputs=[
            answer_output,
            sources_output,
            source_count,
            results_section
        ],
        show_progress="full"
    )

    # ---------- EXAMPLE QUESTIONS ----------
    example_1.click(
        fn=lambda: (
            "What does research show about semaglutide and weight loss?"
        ),
        outputs=question_input
    )

    example_2.click(
        fn=lambda: (
            "How does tirzepatide affect blood sugar control "
            "in type 2 diabetes?"
        ),
        outputs=question_input
    )

    example_3.click(
        fn=lambda: (
            "How do semaglutide and tirzepatide compare "
            "in type 2 diabetes?"
        ),
        outputs=question_input
    )

print("Gradio Interface created successfully!")

Gradio Interface created successfully!


In [105]:
# inbrowser=True: Opens the interface in default browser
demo.launch(inbrowser=True, share=False)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
